# Manuscript benchmark: 3D tet L1 (with L2 pre) on the full real DVF

Companion to `03_benchmark_2d_real_full.ipynb`. This notebook runs the same `L2-multipass -> L1-polish` recipe but on the **full 3D volume** at once, using the manuscript's **6-tetrahedra signed-volume** check as the constraint (the existing `dvfopt.core.iterative3d.iterative_3d` solver uses central-difference Jdet, not the tet check, so the windowed 6-tet solver is inlined here so the reader sees exactly what was run).

**Pipeline.** Starting from the full `(3, 528, 320, 456)` field:

1. **L2 outer loop.** While `n_neg_tet > 0`:
   - find the worst-tet voxel,
   - take the connected component of folded cells that contains it,
   - clip to a bounding box (with 1-voxel positive border, voxel-count cap),
   - run L2-objective SLSQP locally with the per-tet `V_t >= tau` constraint set,
   - splice the solution back into the global field.
2. **L1 polish.** Find connected components of *touched* cells from the L2 phase. For each, re-solve the sub-window with the smoothed-L1 objective, accept only if no new folds appear and L1 strictly drops.
3. Trajectory CSV is appended at every outer iteration so the run is fully resumable on kernel kill.

**Outputs** under `notebooks/manuscript/output/3d_real_full/`:

- `trajectory.csv`  -- one row per outer-loop step (L2 + L1 stages).
- `checkpoint.npz`  -- current corrected `(3, D, H, W)` volume.
- `summary.json`, `summary_*.{pdf,png}`.

## Setup

In [ ]:
import os, sys, time, json, traceback
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, NonlinearConstraint
from scipy.ndimage import label as cc_label

from dvfopt import DEFAULT_PARAMS

THRESHOLD = DEFAULT_PARAMS['threshold']  # 0.01
EPS_L1 = 1e-4

# Window-size cap for the local SLSQP problem (in *cells*, not voxels).
# A cell is a unit cube of 8 voxel corners; an SxSxS cell window has
# (S+1)**3 voxel corners and 6*S**3 per-tet constraints. Cap chosen so a
# typical window's SLSQP run takes O(seconds).
MAX_WINDOW_CELLS = 800           # max total cells in a local window
MAX_WINDOW_PER_AXIS = 12         # max along any single axis
L2_MAX_OUTER = 5000              # outer-loop iteration cap (driven to n_neg=0)
L2_MAX_MINIMIZE_ITER = 300       # per-window SLSQP iter cap
L2_STALL_BUDGET = 20             # consecutive no-improvement iters before window growth
L1_POLISH_MAX_ITER = 200         # per-component L1 SLSQP cap
L1_TOUCH_TOL = 1e-3
L1_BORDER = 1
L1_COMPONENT_MAX_CELLS = 600     # skip polish on enormous components

CHECKPOINT_EVERY = 25            # outer-loop iters between full checkpoints

SMOKE_TEST = False               # crop to a small ROI for sanity-check
SMOKE_ROI = None                 # tuple (z0,z1,y0,y1,x0,x1) when SMOKE_TEST

OUTPUT_DIR = os.path.abspath('output/3d_real_full')
os.makedirs(OUTPUT_DIR, exist_ok=True)
TRAJ_PATH = os.path.join(OUTPUT_DIR, 'trajectory.csv')
CKPT_PATH = os.path.join(OUTPUT_DIR, 'checkpoint.npz')
LOG_PATH = os.path.join(OUTPUT_DIR, 'run.log')

DATA_PATH = os.path.abspath(
    os.path.join('..', '..', 'data',
                 'corrected_correspondences_count_touching',
                 'registered_output', 'deformation3d.npy'))

print(f'THRESHOLD = {THRESHOLD},  EPS_L1 = {EPS_L1}')

## 6-tetrahedra geometry (inlined from notebooks 17/18)

In [ ]:
CUBE_CORNERS = np.array([
    [0,0,0], [0,0,1], [0,1,0], [0,1,1],
    [1,0,0], [1,0,1], [1,1,0], [1,1,1],
], dtype=np.int8)
TET_INDICES = np.array([
    [0,1,3,7], [0,1,5,7], [0,2,3,7],
    [0,2,6,7], [0,4,5,7], [0,4,6,7],
], dtype=np.int8)


def warp_corners(phi):
    D_, H_, W_ = phi.shape[1:]
    zz, yy, xx = np.mgrid[:D_, :H_, :W_]
    return np.stack([xx + phi[2], yy + phi[1], zz + phi[0]], axis=-1)


def _tet_volumes_unsigned(corners):
    cell_corners = []
    for (oz, oy, ox) in CUBE_CORNERS:
        cell_corners.append(corners[oz:corners.shape[0]-1+oz,
                                     oy:corners.shape[1]-1+oy,
                                     ox:corners.shape[2]-1+ox])
    cell_corners = np.stack(cell_corners, axis=0)
    raw = np.empty((6,) + cell_corners.shape[1:-1], dtype=corners.dtype)
    for ti, (ia, ib, ic, id_) in enumerate(TET_INDICES):
        a = cell_corners[ia]; b = cell_corners[ib]
        c = cell_corners[ic]; d = cell_corners[id_]
        ab = b - a; ac = c - a; ad = d - a
        cx = ac[..., 1] * ad[..., 2] - ac[..., 2] * ad[..., 1]
        cy = ac[..., 2] * ad[..., 0] - ac[..., 0] * ad[..., 2]
        cz = ac[..., 0] * ad[..., 1] - ac[..., 1] * ad[..., 0]
        raw[ti] = ab[..., 0] * cx + ab[..., 1] * cy + ab[..., 2] * cz
    return raw


_REF_SIGN = np.sign(_tet_volumes_unsigned(warp_corners(np.zeros((3,2,2,2))))[:, 0, 0, 0]).astype(np.float64)


def tet_signed_volumes(phi):
    raw = _tet_volumes_unsigned(warp_corners(phi))
    return _REF_SIGN[:, None, None, None] * raw / 6.0


def tet_min_per_cell(phi):
    return tet_signed_volumes(phi).min(axis=0)

## Load the real DVF

In [ ]:
phi_full = np.load(DATA_PATH)   # (3, D, H, W) with [dz, dy, dx]
if SMOKE_TEST and SMOKE_ROI is not None:
    (z0_,z1_,y0_,y1_,x0_,x1_) = SMOKE_ROI
    phi_full = phi_full[:, z0_:z1_, y0_:y1_, x0_:x1_].copy()
    print(f'SMOKE_TEST ROI applied: {SMOKE_ROI} -> shape {phi_full.shape}')
D, H, W = phi_full.shape[1:]
print(f'phi_full shape : {phi_full.shape}')

t0 = time.time()
V0 = tet_signed_volumes(phi_full)
n_neg0 = int((V0 <= 0).sum())
min_t0 = float(V0.min())
cells_folded0 = int((V0.min(axis=0) <= 0).sum())
print(f'initial 6-tet check  ({time.time()-t0:.1f}s):')
print(f'  n_neg_tet           : {n_neg0}')
print(f'  min tet volume      : {min_t0:+.4f}')
print(f'  cells with any flip : {cells_folded0}  (of {V0.shape[1]*V0.shape[2]*V0.shape[3]} total)')

## Inline windowed 3D-tet SLSQP solver

`solve_local_l2_3d` and `solve_local_l1_3d` run SLSQP on one sub-window of the field, with the outer 1-cell layer treated as the frozen boundary. The driver loops over folded connected components, picking the worst-tet's component each iteration.

In [ ]:
def _window_pack_unpack(phi_window, interior_mask):
    """Return (z_init, unpack_fn) for SLSQP on a 3D sub-window.
    Variables = phi values at interior cells (where ``interior_mask`` is True).
    """
    int_idx = np.argwhere(interior_mask)
    n_int = len(int_idx)
    iz, iy, ix = int_idx[:, 0], int_idx[:, 1], int_idx[:, 2]

    def pack(phi):
        return np.concatenate([
            phi[2][iz, iy, ix], phi[1][iz, iy, ix], phi[0][iz, iy, ix]
        ])

    def unpack(z, base):
        out = base.copy()
        out[2][iz, iy, ix] = z[:n_int]
        out[1][iz, iy, ix] = z[n_int:2*n_int]
        out[0][iz, iy, ix] = z[2*n_int:]
        return out

    return pack, unpack, n_int


def solve_local_l2_3d(phi_win, phi_anchor_win, interior_mask,
                       threshold=THRESHOLD, max_iter=L2_MAX_MINIMIZE_ITER):
    pack, unpack, n_int = _window_pack_unpack(phi_win, interior_mask)
    if n_int == 0:
        return phi_win.copy(), {'nit': 0, 'success': True, 'status': 0}
    z_init = pack(phi_win)
    z_anchor = pack(phi_anchor_win)
    def obj(z):
        d = z - z_anchor
        return 0.5 * float(np.dot(d, d)), d
    def constr(z):
        return tet_signed_volumes(unpack(z, phi_win)).flatten()
    nl = NonlinearConstraint(constr, lb=threshold, ub=np.inf)
    res = minimize(obj, z_init, jac=True, method='SLSQP', constraints=[nl],
                    options={'maxiter': max_iter, 'disp': False})
    return unpack(res.x, phi_win), {'nit': int(res.nit),
                                     'success': bool(res.success),
                                     'status': int(res.status)}


def solve_local_l1_3d(phi_win, phi_anchor_win, interior_mask,
                       threshold=THRESHOLD, eps=EPS_L1,
                       max_iter=L1_POLISH_MAX_ITER):
    pack, unpack, n_int = _window_pack_unpack(phi_win, interior_mask)
    if n_int == 0:
        return phi_win.copy(), {'nit': 0, 'success': True, 'status': 0}
    z_init = pack(phi_win)
    z_anchor = pack(phi_anchor_win)
    def obj(z):
        d = z - z_anchor
        s = np.sqrt(d * d + eps * eps)
        return float(s.sum()), d / s
    def constr(z):
        return tet_signed_volumes(unpack(z, phi_win)).flatten()
    nl = NonlinearConstraint(constr, lb=threshold, ub=np.inf)
    res = minimize(obj, z_init, jac=True, method='SLSQP', constraints=[nl],
                    options={'maxiter': max_iter, 'ftol': 1e-9, 'disp': False})
    return unpack(res.x, phi_win), {'nit': int(res.nit),
                                     'success': bool(res.success),
                                     'status': int(res.status)}

In [ ]:
def _shrink_to_cap(z0, z1, y0, y1, x0, x1, cap_per_axis, cap_total):
    """Clip a cell-window bbox so each axis <= ``cap_per_axis`` and the
    total cell count <= ``cap_total``. Shrinks evenly from both sides.
    """
    def shrink_axis(a0, a1, cap):
        if a1 - a0 <= cap:
            return a0, a1
        ctr = (a0 + a1) // 2
        half = cap // 2
        return max(a0, ctr - half), min(a1, ctr - half + cap)
    z0, z1 = shrink_axis(z0, z1, cap_per_axis)
    y0, y1 = shrink_axis(y0, y1, cap_per_axis)
    x0, x1 = shrink_axis(x0, x1, cap_per_axis)
    # Total-volume cap: shrink the longest axis 1 at a time.
    while (z1-z0)*(y1-y0)*(x1-x0) > cap_total:
        spans = [(z1-z0, 'z'), (y1-y0, 'y'), (x1-x0, 'x')]
        spans.sort(reverse=True)
        ax = spans[0][1]
        if ax == 'z' and z1 - z0 > 2:
            if (z0 + z1) % 2 == 0: z1 -= 1
            else: z0 += 1
        elif ax == 'y' and y1 - y0 > 2:
            if (y0 + y1) % 2 == 0: y1 -= 1
            else: y0 += 1
        elif ax == 'x' and x1 - x0 > 2:
            if (x0 + x1) % 2 == 0: x1 -= 1
            else: x0 += 1
        else:
            break
    return z0, z1, y0, y1, x0, x1


def build_window_around(phi, focus_cell, *, border=1,
                         cap_per_axis=MAX_WINDOW_PER_AXIS,
                         cap_total=MAX_WINDOW_CELLS,
                         labels=None, cell_fold_mask=None):
    """Build a SLSQP sub-window centered on the connected component of folded
    cells that contains ``focus_cell`` (cz, cy, cx). Adds a ``border``-cell
    positive frame on each side, then clips to ``cap_*`` constraints.

    Returns ``(z0, z1, y0, y1, x0, x1)`` in *cell* coordinates. Voxel-corner
    window is then ``[z0:z1+1, y0:y1+1, x0:x1+1]``.
    """
    cz, cy, cx = focus_cell
    if cell_fold_mask is None or labels is None:
        V = tet_signed_volumes(phi)
        cell_fold_mask = V.min(axis=0) <= THRESHOLD - 1e-9
        labels, _ = cc_label(cell_fold_mask)
    cid = labels[cz, cy, cx]
    if cid == 0:
        # Single-cell window around the focus.
        z0, z1 = cz, cz + 1
        y0, y1 = cy, cy + 1
        x0, x1 = cx, cx + 1
    else:
        comp = np.argwhere(labels == cid)
        z0, y0, x0 = comp.min(axis=0)
        z1, y1, x1 = comp.max(axis=0) + 1
    # Apply border (in cell units).
    Dc, Hc, Wc = labels.shape
    z0 = max(0, z0 - border); z1 = min(Dc, z1 + border)
    y0 = max(0, y0 - border); y1 = min(Hc, y1 + border)
    x0 = max(0, x0 - border); x1 = min(Wc, x1 + border)
    # Cap (also enforces minimum 2x2x2 implicitly when capped).
    z0, z1, y0, y1, x0, x1 = _shrink_to_cap(z0, z1, y0, y1, x0, x1,
                                              cap_per_axis, cap_total)
    return int(z0), int(z1), int(y0), int(y1), int(x0), int(x1)

## Outer L2 loop

In [ ]:
def cells_window_to_voxel_slice(z0, z1, y0, y1, x0, x1):
    """Cell-bbox -> voxel-corner sub-array slice. (z1+1, y1+1, x1+1) inclusive."""
    return slice(z0, z1 + 1), slice(y0, y1 + 1), slice(x0, x1 + 1)


def field_metrics_3d(phi_now, phi_anchor):
    V = tet_signed_volumes(phi_now)
    return {
        'n_neg_tet': int((V <= 0).sum()),
        'min_tet': float(V.min()),
        'cells_folded': int((V.min(axis=0) <= 0).sum()),
        'L1': float(np.abs(phi_now - phi_anchor).sum()),
        'L2': float(np.linalg.norm(phi_now - phi_anchor)),
    }

In [ ]:
TRAJ_COLUMNS = [
    'outer_iter', 'stage',
    'win_z0', 'win_z1', 'win_y0', 'win_y1', 'win_x0', 'win_x1',
    'win_cells', 'win_interior_cells',
    'slsqp_nit', 'slsqp_success', 'slsqp_status',
    'n_neg_tet_before', 'n_neg_tet_after',
    'min_tet_before', 'min_tet_after',
    'L1_before', 'L1_after', 'L2_before', 'L2_after',
    't_seconds',
]


def init_csv():
    if not os.path.exists(TRAJ_PATH):
        with open(TRAJ_PATH, 'w') as f:
            f.write(','.join(TRAJ_COLUMNS) + '\n')


def append_traj(row):
    parts = []
    for c in TRAJ_COLUMNS:
        v = row.get(c, '')
        if v is None:
            parts.append('')
        elif isinstance(v, float):
            parts.append(f'{v:.6g}')
        elif isinstance(v, bool):
            parts.append('True' if v else 'False')
        else:
            parts.append(str(v))
    with open(TRAJ_PATH, 'a') as f:
        f.write(','.join(parts) + '\n')


def save_checkpoint(arr, outer_iter):
    tmp = CKPT_PATH + '.tmp'
    np.savez_compressed(tmp, phi_corrected=arr, outer_iter=outer_iter)
    os.replace(tmp, CKPT_PATH)


def load_checkpoint(default):
    if not os.path.exists(CKPT_PATH):
        return default.copy(), 0
    with np.load(CKPT_PATH) as data:
        arr = data['phi_corrected']
        oi = int(data['outer_iter']) if 'outer_iter' in data.files else 0
        if arr.shape == default.shape:
            return arr.copy(), oi
    return default.copy(), 0


def log_line(msg):
    print(msg, flush=True)
    with open(LOG_PATH, 'a') as f:
        f.write(msg + '\n')


init_csv()

In [ ]:
phi = phi_full.copy()
phi_anchor = phi_full.copy()
phi, start_outer = load_checkpoint(phi)
log_line(f'[start] {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}  '
          f'resume_from_outer={start_outer}')

stall = 0
last_n_neg = None
window_growth_factor = 1.0
outer_iter = start_outer

while outer_iter < L2_MAX_OUTER:
    V = tet_signed_volumes(phi)
    n_neg = int((V <= 0).sum())
    if n_neg == 0:
        log_line(f'[L2-done] outer={outer_iter}  n_neg_tet=0')
        break
    cell_fold_mask = V.min(axis=0) <= THRESHOLD - 1e-9
    if not cell_fold_mask.any():
        log_line(f'[L2-done] outer={outer_iter}  all min_tet >= tau-eps')
        break
    labels, _ = cc_label(cell_fold_mask)
    # Pick the cell containing the worst tet.
    argmin_flat = int(V.argmin())
    ti, cz, cy, cx = np.unravel_index(argmin_flat, V.shape)
    cap_per_axis = max(2, int(MAX_WINDOW_PER_AXIS * window_growth_factor))
    cap_total = int(MAX_WINDOW_CELLS * (window_growth_factor ** 3))
    z0, z1, y0, y1, x0, x1 = build_window_around(
        phi, (cz, cy, cx), border=1,
        cap_per_axis=cap_per_axis, cap_total=cap_total,
        labels=labels, cell_fold_mask=cell_fold_mask)
    sz, sy, sx = z1 - z0, y1 - y0, x1 - x0
    win_cells = sz * sy * sx
    vsz = sz + 1; vsy = sy + 1; vsx = sx + 1
    # Window slices (voxel corners).
    vz, vy, vx = cells_window_to_voxel_slice(z0, z1, y0, y1, x0, x1)
    phi_win = phi[:, vz, vy, vx].copy()
    phi_anchor_win = phi_anchor[:, vz, vy, vx]
    # Interior mask in voxel-corner space: freeze the outer 1-layer.
    interior_mask = np.zeros((vsz, vsy, vsx), dtype=bool)
    if vsz > 2 and vsy > 2 and vsx > 2:
        interior_mask[1:-1, 1:-1, 1:-1] = True
    win_interior_cells = int(interior_mask.sum())
    # SLSQP local solve.
    t0 = time.time()
    L1b = float(np.abs(phi - phi_anchor).sum())
    L2b = float(np.linalg.norm(phi - phi_anchor))
    min_tet_before = float(V.min())
    try:
        phi_win_new, info = solve_local_l2_3d(
            phi_win, phi_anchor_win, interior_mask,
            threshold=THRESHOLD, max_iter=L2_MAX_MINIMIZE_ITER)
    except Exception as exc:
        log_line(f'[ERR  ] outer={outer_iter}  SLSQP failure: {type(exc).__name__}: {exc}')
        info = {'nit': 0, 'success': False, 'status': -1}
        phi_win_new = phi_win
    phi[:, vz, vy, vx] = phi_win_new
    V2 = tet_signed_volumes(phi)
    n_neg_after = int((V2 <= 0).sum())
    min_tet_after = float(V2.min())
    L1a = float(np.abs(phi - phi_anchor).sum())
    L2a = float(np.linalg.norm(phi - phi_anchor))
    elapsed = time.time() - t0
    row = {
        'outer_iter': outer_iter, 'stage': 'L2',
        'win_z0': z0, 'win_z1': z1, 'win_y0': y0, 'win_y1': y1,
        'win_x0': x0, 'win_x1': x1,
        'win_cells': win_cells, 'win_interior_cells': win_interior_cells,
        'slsqp_nit': info['nit'], 'slsqp_success': info['success'],
        'slsqp_status': info['status'],
        'n_neg_tet_before': n_neg, 'n_neg_tet_after': n_neg_after,
        'min_tet_before': min_tet_before, 'min_tet_after': min_tet_after,
        'L1_before': L1b, 'L1_after': L1a, 'L2_before': L2b, 'L2_after': L2a,
        't_seconds': elapsed,
    }
    append_traj(row)
    if last_n_neg is None or n_neg_after < last_n_neg:
        stall = 0
        window_growth_factor = max(1.0, window_growth_factor / 1.5)
    else:
        stall += 1
        if stall >= L2_STALL_BUDGET:
            window_growth_factor = min(4.0, window_growth_factor * 1.5)
            stall = 0
            log_line(f'[grow ] outer={outer_iter}  window_growth_factor -> {window_growth_factor:.2f}')
    last_n_neg = n_neg_after
    if outer_iter % 10 == 0:
        log_line(f'[L2  ] outer={outer_iter:>4d}  n_neg_tet {n_neg}->{n_neg_after}  '
                  f'min_tet {min_tet_before:+.4f}->{min_tet_after:+.4f}  '
                  f'L1 {L1b:.2f}->{L1a:.2f}  win={sz}x{sy}x{sx}  '
                  f'slsqp_nit={info["nit"]} ok={info["success"]}  t={elapsed:.1f}s')
    if outer_iter % CHECKPOINT_EVERY == 0 and outer_iter > start_outer:
        save_checkpoint(phi, outer_iter)
    outer_iter += 1

save_checkpoint(phi, outer_iter)
phi_after_l2 = phi.copy()
log_line(f'[L2-end] outer={outer_iter}  n_neg_tet={int((tet_signed_volumes(phi)<=0).sum())}')

## L1 polish phase

Find connected components of cells touched by the L2 phase, run SLSQP with the smoothed-L1 objective on each window, accept only if it strictly drops the L1 norm without re-introducing folds.

In [ ]:
def run_l1_polish_3d(phi_l2, phi_anchor, outer_offset=0):
    diff_mag = np.abs(phi_l2 - phi_anchor).max(axis=0)
    # Treat 'touched' at the voxel-corner level: a cell is 'touched'
    # if any of its 8 corners moved.
    Dc, Hc, Wc = diff_mag.shape[0] - 1, diff_mag.shape[1] - 1, diff_mag.shape[2] - 1
    cell_touched = np.zeros((Dc, Hc, Wc), dtype=bool)
    for (oz, oy, ox) in CUBE_CORNERS:
        cell_touched |= diff_mag[oz:Dc+oz, oy:Hc+oy, ox:Wc+ox] > L1_TOUCH_TOL
    if not cell_touched.any():
        log_line('[L1  ] no touched cells -- L1 polish has nothing to do')
        return phi_l2.copy()
    labels, n_comp = cc_label(cell_touched)
    log_line(f'[L1  ] touched cells -> {n_comp} connected components')
    phi_polished = phi_l2.copy()
    outer_iter = outer_offset
    accepted = 0
    rejected = 0
    skipped_large = 0
    for cid in range(1, n_comp + 1):
        comp = np.argwhere(labels == cid)
        z0, y0, x0 = comp.min(axis=0)
        z1, y1, x1 = comp.max(axis=0) + 1
        # Add border.
        z0 = max(0, z0 - L1_BORDER); y0 = max(0, y0 - L1_BORDER); x0 = max(0, x0 - L1_BORDER)
        z1 = min(Dc, z1 + L1_BORDER); y1 = min(Hc, y1 + L1_BORDER); x1 = min(Wc, x1 + L1_BORDER)
        # Cap.
        z0, z1, y0, y1, x0, x1 = _shrink_to_cap(
            z0, z1, y0, y1, x0, x1,
            cap_per_axis=MAX_WINDOW_PER_AXIS,
            cap_total=L1_COMPONENT_MAX_CELLS)
        win_cells = (z1-z0) * (y1-y0) * (x1-x0)
        if win_cells <= 0:
            rejected += 1
            continue
        if (z1-z0) > MAX_WINDOW_PER_AXIS or (y1-y0) > MAX_WINDOW_PER_AXIS or (x1-x0) > MAX_WINDOW_PER_AXIS:
            skipped_large += 1
            continue
        vz, vy, vx = cells_window_to_voxel_slice(z0, z1, y0, y1, x0, x1)
        phi_win_l2 = phi_polished[:, vz, vy, vx].copy()
        phi_anchor_win = phi_anchor[:, vz, vy, vx]
        vsz, vsy, vsx = z1-z0+1, y1-y0+1, x1-x0+1
        interior_mask = np.zeros((vsz, vsy, vsx), dtype=bool)
        if vsz > 2 and vsy > 2 and vsx > 2:
            interior_mask[1:-1, 1:-1, 1:-1] = True
        if not interior_mask.any():
            rejected += 1
            continue
        L1_before_global = float(np.abs(phi_polished - phi_anchor).sum())
        L1_before_win = float(np.abs(phi_win_l2 - phi_anchor_win).sum())
        t0 = time.time()
        try:
            phi_win_new, info = solve_local_l1_3d(
                phi_win_l2, phi_anchor_win, interior_mask,
                threshold=THRESHOLD, max_iter=L1_POLISH_MAX_ITER)
        except Exception as exc:
            log_line(f'[ERR  ] L1 polish cid={cid} failed: {type(exc).__name__}: {exc}')
            rejected += 1
            continue
        L1_after_win = float(np.abs(phi_win_new - phi_anchor_win).sum())
        # Build candidate and check global feasibility.
        phi_candidate = phi_polished.copy()
        phi_candidate[:, vz, vy, vx] = phi_win_new
        Vc = tet_signed_volumes(phi_candidate)
        n_neg_after = int((Vc <= 0).sum())
        elapsed = time.time() - t0
        # Accept if strictly improves L1 and stays feasible.
        if n_neg_after == 0 and L1_after_win < L1_before_win - 1e-9:
            phi_polished = phi_candidate
            L1_after_global = float(np.abs(phi_polished - phi_anchor).sum())
            accepted += 1
            stage_label = 'L1_accept'
        else:
            L1_after_global = L1_before_global
            rejected += 1
            stage_label = 'L1_reject'
        row = {
            'outer_iter': outer_iter, 'stage': stage_label,
            'win_z0': z0, 'win_z1': z1, 'win_y0': y0, 'win_y1': y1,
            'win_x0': x0, 'win_x1': x1,
            'win_cells': win_cells,
            'win_interior_cells': int(interior_mask.sum()),
            'slsqp_nit': info['nit'], 'slsqp_success': info['success'],
            'slsqp_status': info['status'],
            'n_neg_tet_before': 0, 'n_neg_tet_after': n_neg_after,
            'min_tet_before': float('nan'), 'min_tet_after': float(Vc.min()),
            'L1_before': L1_before_global, 'L1_after': L1_after_global,
            'L2_before': float('nan'), 'L2_after': float('nan'),
            't_seconds': elapsed,
        }
        append_traj(row)
        outer_iter += 1
        if accepted % 25 == 0 and accepted > 0:
            save_checkpoint(phi_polished, outer_iter)
    log_line(f'[L1-end] components: {accepted} accepted, {rejected} rejected, '
              f'{skipped_large} skipped (too large)')
    save_checkpoint(phi_polished, outer_iter)
    return phi_polished


phi_final = run_l1_polish_3d(phi_after_l2, phi_anchor, outer_offset=outer_iter)
log_line(f'[end  ] {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}')

## Aggregate stats and plots

In [ ]:
df = pd.read_csv(TRAJ_PATH)
df_l2 = df[df['stage'] == 'L2'].copy()
df_l1_accept = df[df['stage'] == 'L1_accept'].copy()
df_l1_reject = df[df['stage'] == 'L1_reject'].copy()

init_metrics = field_metrics_3d(phi_full, phi_full)
final_metrics = field_metrics_3d(phi_final, phi_full)
after_l2_metrics = field_metrics_3d(phi_after_l2, phi_full)

print(f'INITIAL : n_neg_tet={init_metrics["n_neg_tet"]}  '
      f'min_tet={init_metrics["min_tet"]:+.4f}  '
      f'cells_folded={init_metrics["cells_folded"]}')
print(f'AFTER L2: n_neg_tet={after_l2_metrics["n_neg_tet"]}  '
      f'min_tet={after_l2_metrics["min_tet"]:+.4f}  '
      f'L1={after_l2_metrics["L1"]:.2f}  L2={after_l2_metrics["L2"]:.2f}')
print(f'AFTER L1: n_neg_tet={final_metrics["n_neg_tet"]}  '
      f'min_tet={final_metrics["min_tet"]:+.4f}  '
      f'L1={final_metrics["L1"]:.2f}  L2={final_metrics["L2"]:.2f}')
if after_l2_metrics['L1'] > 0:
    drop_pct = 100.0 * (after_l2_metrics['L1'] - final_metrics['L1']) / after_l2_metrics['L1']
    print(f'L1 drop after polish: {drop_pct:.2f}%')
print()
print(f'outer L2 iterations: {len(df_l2)}')
print(f'L1 polish components: {len(df_l1_accept)} accepted, {len(df_l1_reject)} rejected')
print(f'total wall time    : {df["t_seconds"].sum():.1f}s = {df["t_seconds"].sum()/3600:.2f} h')

In [ ]:
FIG_DIR = OUTPUT_DIR

# (1) fold-count trajectory
if len(df_l2) > 0:
    fig, ax = plt.subplots(figsize=(11, 3.6), constrained_layout=True)
    ax.plot(df_l2['outer_iter'], df_l2['n_neg_tet_after'],
             lw=1.0, color='#c62828')
    ax.set_yscale('symlog', linthresh=1)
    ax.set_xlabel('outer iteration'); ax.set_ylabel('n_neg_tet (symlog)')
    ax.set_title('Folded-tet count vs outer iteration (L2 phase)')
    fig.savefig(os.path.join(FIG_DIR, 'fold_count_trajectory.png'), dpi=200, bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, 'fold_count_trajectory.pdf'), bbox_inches='tight')
    plt.show()

# (2) L1 / L2 norm trajectories on twin axes
if len(df_l2) > 0:
    fig, ax = plt.subplots(figsize=(11, 3.6), constrained_layout=True)
    color1 = '#1976d2'; color2 = '#e65100'
    ax.plot(df_l2['outer_iter'], df_l2['L1_after'], lw=1.0, color=color1, label='L1')
    ax.set_xlabel('outer iteration'); ax.set_ylabel('L1 norm', color=color1)
    ax2 = ax.twinx()
    ax2.plot(df_l2['outer_iter'], df_l2['L2_after'], lw=1.0, color=color2, label='L2')
    ax2.set_ylabel('L2 norm', color=color2)
    ax.set_title('L1 and L2 norms of correction vs outer iteration')
    fig.savefig(os.path.join(FIG_DIR, 'l1_l2_norm_trajectory.png'), dpi=200, bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, 'l1_l2_norm_trajectory.pdf'), bbox_inches='tight')
    plt.show()

# (3) spatial distribution of initial folds (projections to xy/xz/yz)
V_init = tet_signed_volumes(phi_full)
fold_mask_init = V_init.min(axis=0) <= 0  # (D, H, W) cells
fig, axes = plt.subplots(1, 3, figsize=(13, 4.0), constrained_layout=True)
axes[0].imshow(fold_mask_init.sum(axis=0), cmap='Reds')
axes[0].set_title('initial fold density (projected along z)'); axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[1].imshow(fold_mask_init.sum(axis=1), cmap='Reds')
axes[1].set_title('initial fold density (projected along y)'); axes[1].set_xlabel('x'); axes[1].set_ylabel('z')
axes[2].imshow(fold_mask_init.sum(axis=2), cmap='Reds')
axes[2].set_title('initial fold density (projected along x)'); axes[2].set_xlabel('y'); axes[2].set_ylabel('z')
fig.savefig(os.path.join(FIG_DIR, 'fold_spatial_distribution.png'), dpi=200, bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'fold_spatial_distribution.pdf'), bbox_inches='tight')
plt.show()

## Save final volume + summary JSON

In [ ]:
final_path = os.path.join(OUTPUT_DIR, 'deformation3d_corrected_3d.npy')
np.save(final_path, phi_final)
print(f'Saved corrected volume to {final_path}')

summary = {
    'data_path': DATA_PATH,
    'shape': list(phi_full.shape),
    'threshold': THRESHOLD, 'eps_l1': EPS_L1,
    'max_window_cells': MAX_WINDOW_CELLS,
    'max_window_per_axis': MAX_WINDOW_PER_AXIS,
    'l2_max_outer': L2_MAX_OUTER,
    'l2_max_minimize_iter': L2_MAX_MINIMIZE_ITER,
    'l1_polish_max_iter': L1_POLISH_MAX_ITER,
    'init_n_neg_tet': init_metrics['n_neg_tet'],
    'init_min_tet': init_metrics['min_tet'],
    'init_cells_folded': init_metrics['cells_folded'],
    'after_l2_n_neg_tet': after_l2_metrics['n_neg_tet'],
    'after_l2_min_tet': after_l2_metrics['min_tet'],
    'after_l2_L1': after_l2_metrics['L1'],
    'after_l2_L2': after_l2_metrics['L2'],
    'after_l1_n_neg_tet': final_metrics['n_neg_tet'],
    'after_l1_min_tet': final_metrics['min_tet'],
    'after_l1_L1': final_metrics['L1'],
    'after_l1_L2': final_metrics['L2'],
    'l2_outer_iterations': int(len(df_l2)),
    'l1_components_accepted': int(len(df_l1_accept)),
    'l1_components_rejected': int(len(df_l1_reject)),
    'total_wall_time_s': float(df['t_seconds'].sum()),
}
with open(os.path.join(OUTPUT_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved summary.json')